# 05: Evaluate LLaVA + LoRA (H3 Ablation)

Loads the LoRA adapter trained in notebook 04 onto the LLaVA backbone and
runs ORA on the synthetic held-out set + a Mind2Web slice. Prints a base-vs-
adapter delta table for the H3 row of the report.

**Runtime:** A100 GPU. **Wallclock:** ~30-40 min. **Compute units:** ~7-9.

In [1]:
# Mount Google Drive. The first run prompts for OAuth; subsequent runs
# in the same session reuse the token automatically.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/grounded_vla')
ROOT.mkdir(parents=True, exist_ok=True)
print('drive root:', ROOT)

Mounted at /content/drive
drive root: /content/drive/MyDrive/grounded_vla


In [2]:
# Verify accelerator. Pro+ should give you A100 most of the time; if you
# see T4, change Runtime -> Change runtime type -> A100 GPU and re-run.
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv'])

CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], returncode=0)

In [3]:
# Clone repo if absent, then always pull to get the latest fixes.
# With an editable install (-e) a pull is all that's needed — no reinstall.
REPO_URL = 'https://github.com/KarthikRed2000/grounded_vla.git'
import os, subprocess
if not os.path.exists('/content/grounded_vla'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/grounded_vla'], check=True)
subprocess.run(['git', '-C', '/content/grounded_vla', 'pull'], check=True)
%cd /content/grounded_vla

/content/grounded_vla


In [4]:
import os
adapter_dir = '/content/drive/MyDrive/grounded_vla/checkpoints/llava-lora-r1'
print(os.listdir(adapter_dir))
# Should show: ['adapter_config.json', 'adapter_model.safetensors', ...]

['checkpoint-25', 'checkpoint-50', 'checkpoint-75', 'README.md', 'adapter_model.safetensors', 'adapter_config.json']


In [6]:
# Install repo + GPU stack. Quiet to keep the cell output sane.
!pip install -q -e .[gpu,data]

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.2 MB/s eta 0:00:00
  Building editable for grounded_vla (pyproject.toml) ... done


In [7]:
import os
WEIGHTS = '/content/drive/MyDrive/grounded_vla/hf_cache'
DATA    = '/content/drive/MyDrive/grounded_vla/data'
ADAPTER = '/content/drive/MyDrive/grounded_vla/checkpoints/llava-lora-r1'
os.environ['HF_HOME'] = WEIGHTS
os.environ['TRANSFORMERS_OFFLINE'] = '1'

In [8]:
# Load LLaVA + adapter via a thin subclass that injects PEFT after load.
from grounded_vla.backends.llava import LLaVABackend

class LLaVAWithLoRA(LLaVABackend):
    name = 'llava-1.6-7b+lora'
    def __init__(self, adapter_dir, **kw):
        super().__init__(**kw)
        self.adapter_dir = adapter_dir
    def _ensure_loaded(self):
        if self._model is not None:
            return
        super()._ensure_loaded()
        from peft import PeftModel
        self._model = PeftModel.from_pretrained(self._model, self.adapter_dir)
        self._model.eval()
        print(f'[LLaVAWithLoRA] adapter loaded from {self.adapter_dir}')

In [12]:
from grounded_vla.agents import ORAAgent
from grounded_vla.backends.base import GenerationConfig
from grounded_vla.data import make_dataset
from grounded_vla.eval import EvalRunner
from pathlib import Path

backend = LLaVAWithLoRA(
    adapter_dir=ADAPTER,
    model_id=f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
    device='cuda',
    quantize='4bit',
)
agent = ORAAgent(backend, GenerationConfig(max_new_tokens=384, temperature=0.1))

RUNS = Path('/content/drive/MyDrive/grounded_vla/runs')

def run(ds_name, dataset):
    out = RUNS / f'ora_lora__{ds_name}'
    res = EvalRunner(agent).evaluate(
        dataset, save_dir=out, checkpoint_every=5, resume=True
    )
    print(f'ora_lora on {ds_name}: '
          f'success={res.task_completion_rate:.3f} '
          f'mean_steps={res.mean_steps:.2f} '
          f'errors={res.error_breakdown}')
    return res

synth = make_dataset({
    'kind': 'jsonl',
    'path':  f'{DATA}/synthetic_sample/synthetic_sample.jsonl',
    'source': 'synthetic',
})
sqa = make_dataset({
    'kind': 'scienceqa',
    'jsonl_path': f'{DATA}/scienceqa/test.jsonl',
    'images_dir': f'{DATA}/scienceqa',
    'limit': 200,
})

_ = run('synthetic',  synth)
_ = run('scienceqa',  sqa)
backend.close()

[05/07/26 01:58:56] INFO     INFO:numexpr.utils:NumExpr defaulting to 12 threads.

The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[LLaVAWithLoRA] adapter loaded from /content/drive/MyDrive/grounded_vla/checkpoints/llava-lora-r1


...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/07/26 02:29:42] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 195 ->                                
                             /content/drive/MyDrive/grounded_vla/runs/ora_lora__scienceqa

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/07/26 02:30:02] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 200 ->                                
                             /content/drive/MyDrive/grounded_vla/runs/ora_lora__scienceqa

ora_lora on scienceqa: success=0.575 mean_steps=1.00 errors={'visual_misgrounding': 85, 'none': 115}


In [13]:
# Side-by-side vs the non-LoRA ORA runs from notebook 03.
import json, glob
base_runs = sorted(glob.glob('/content/drive/MyDrive/grounded_vla/runs/ora_llava__*'))
lora_runs = sorted(glob.glob('/content/drive/MyDrive/grounded_vla/runs/ora_lora__*'))

def load(p):
    return json.loads(open(f'{p}/summary.json').read())

print(f'{"dataset":12s} {"ora-base":>10s} {"ora+lora":>10s} {"delta":>8s}')
for lp in lora_runs:
    ds = lp.rsplit('__', 1)[-1]
    base = next((p for p in base_runs if p.endswith(ds)), None)
    if not base:
        continue
    b = load(base)['task_completion_rate']
    l = load(lp)['task_completion_rate']
    print(f'{ds:12s} {b:10.3f} {l:10.3f} {l-b:+8.3f}')

dataset        ora-base   ora+lora    delta
scienceqa         0.565      0.575   +0.010
synthetic         0.290      0.920   +0.630


## Interpreting the H3 result

- **Positive delta on synthetic, smaller on Mind2Web:** LoRA closed the in-domain
  gap; cross-domain transfer is partial. Useful talking point in the report.
- **Negative delta:** likely overfitting on a tiny synthetic set. Either grow the
  synthetic corpus to ~200 reviewed examples, or shorten training (1 epoch).
- **No change:** LoRA rank may be too small (try `r=32`) or the action format in
  the synthetic dataset doesn't match the agent's prompt template.

Either way, the result goes into the H3 row of the results table.